# Nados AI v1.1 — خادم الاستدلال السحابي (محصّن)

شغّل الخلايا بالترتيب. أي خلية فشلت أعد تشغيلها. في النهاية ستحصل على **رابط عام 🌍** يستخدمه موقع Nados.

**مهم**: إذا أعاد كولاب تشغيل الجلسة، أعد تشغيل كل الخلايا من الأول (الملفات تُمسح).


In [ ]:
#@title 1) تثبيت llama.cpp وcloudflared (محصّنة)
import subprocess, os, glob

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print('STDERR:', result.stderr[:300])
    return result

if not os.path.exists('llama-server'):
    print('downloading llama.cpp...')
    run('wget -q https://github.com/ggml-org/llama.cpp/releases/download/b4458/llama-b4458-bin-ubuntu-x64.zip -O llama.zip')
    run('unzip -q -o llama.zip -d llama-bin')
    matches = glob.glob('llama-bin/**/llama-server', recursive=True)
    if matches:
        run("cp '" + matches[0] + "' llama-server")
    else:
        print('llama-server غير موجود في الضغط — الملفات:')
        print(run('find llama-bin -type f | head -15').stdout)

if not os.path.exists('cloudflared'):
    run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared')

run('chmod +x llama-server cloudflared')
ok = os.path.exists('llama-server') and os.path.exists('cloudflared')
print('llama-server:', os.path.exists('llama-server'), '| cloudflared:', os.path.exists('cloudflared'))
print('التثبيت مكتمل — انتقل للخلية 2' if ok else 'فشل — الصق لي المخرجات')


In [ ]:
#@title 2) تنزيل النموذج الأساسي والمحوّل المدرَّب
MODEL_SIZE = 'mini' #@param ['mini', 'full']
import os
if MODEL_SIZE == 'mini':
    if not os.path.exists('base.gguf'):
        !wget -q https://huggingface.co/bartowski/gemma-2-2b-it-GGUF/resolve/main/gemma-2-2b-it-Q4_K_M.gguf -O base.gguf
    if not os.path.exists('nados-lora.gguf'):
        !wget -q https://huggingface.co/noore7xd/nados-models/resolve/main/nados-v1-1-mini-lora.gguf -O nados-lora.gguf
else:
    if not os.path.exists('base.gguf'):
        !wget -q https://huggingface.co/bartowski/gemma-2-9b-it-GGUF/resolve/main/gemma-2-9b-it-Q4_K_M.gguf -O base.gguf
    if not os.path.exists('nados-lora.gguf'):
        !wget -q https://huggingface.co/noore7xd/nados-models/resolve/main/nados-v1-1-lora.gguf -O nados-lora.gguf
print('base:', round(os.path.getsize('base.gguf')/(1024**3), 2), 'GB | lora:', round(os.path.getsize('nados-lora.gguf')/(1024**2), 0), 'MB')
print('النماذج جاهزة — انتقل للخلية 3')


In [ ]:
#@title 3) تشغيل خادم Nados v1.1 + النفق العام (محصّنة)
import subprocess, time, re, os

if not os.path.exists('base.gguf') or not os.path.exists('nados-lora.gguf'):
    raise RuntimeError('النماذج غير موجودة — أعد تشغيل الخلية 2 أولاً')
if not os.path.exists('llama-server'):
    raise RuntimeError('llama-server غير موجود — أعد تشغيل الخلية 1 أولاً')

server = subprocess.Popen(['./llama-server', '-m', 'base.gguf', '--lora', 'nados-lora.gguf', '--host', '0.0.0.0', '--port', '8080', '--threads', '2'], stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(15)
health = subprocess.run(['curl', '-s', 'http://localhost:8080/health'], capture_output=True, text=True)
print('الخادم المحلي:', 'يعمل' if 'ok' in health.stdout.lower() else 'فشل: ' + health.stdout[:150])

tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8080', '--no-autoupdate'], stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)

url = None
for attempt in range(6):
    time.sleep(10)
    log = open('tunnel.log').read()
    match = re.search(r'https://[a-z0-9-]+[.]trycloudflare[.]com', log)
    if match:
        url = match.group(0)
        break

if url:
    print('=' * 60)
    print('الرابط العام لنموذج Nados v1.1:')
    print(url)
    print('=' * 60)
else:
    print('لم يظهر الرابط — tunnel.log:')
    print(open('tunnel.log').read()[:600])


In [ ]:
#@title 4) اختبار النموذج المدرَّب (اختياري)
import subprocess, json
question = 'ما فوائد الاختبارات الآلية؟' #@param {type:'string'}
body = json.dumps({'messages': [{'role': 'user', 'content': question}], 'max_tokens': 300})
result = subprocess.run(['curl', '-s', '-X', 'POST', 'http://localhost:8080/v1/chat/completions', '-H', 'Content-Type: application/json', '-d', body], capture_output=True, text=True)
try:
    reply = json.loads(result.stdout)
    print('إجابة Nados v1.1:', reply['choices'][0]['message']['content'][:500])
except Exception:
    print('raw:', result.stdout[:400])
